# kafka-mirror: a live DR failover / fail-back run

This notebook drives the exact same active/passive DR scenario documented in
[`docs/dr-runbook.md`](../docs/dr-runbook.md) against a **live** cluster and
records the real output of every step.

It doesn't reimplement any logic. It shells out to the same
`scripts/status.sh`, `scripts/failover.sh`, `scripts/start-failback-resync.sh`
and `scripts/complete-failback.sh` used from the command line, plus a few
`kubectl exec` / `kubectl logs` calls to show what the producer, consumer and
MirrorMaker 2 are actually doing. This keeps the notebook and the CLI scripts
as a single source of truth.

What it shows, in order:

1. Baseline: `primary-cluster` active, `mm2-primary-to-dr` mirroring, a
   producer/consumer pair reading and writing `drc-demo-topic`.
2. Triggering a failover (`failover.sh`) and watching the producer/consumer
   reconnect to `dr-cluster` without a restart.
3. `dr-cluster` serving fresh writes on its own, with no mirror running.
4. Starting the fail-back resync (`start-failback-resync.sh`) and watching
   `dr-cluster`'s new data get copied back to `primary-cluster`.
5. Completing the fail-back (`complete-failback.sh`) and watching the
   consumer reconnect to `primary-cluster` — including the duplicate records
   the resync reintroduces, and the client's own idempotent dedup dropping
   every one of them.

Re-running all cells performs a real failover/fail-back cycle against
whatever cluster is currently reachable via `microk8s kubectl`.


In [1]:
import subprocess
import time
import sys

REPO = "/home/ozcan/Documents/kafka-mirror"


def run(cmd, timeout=180):
    """Run a shell command from the repo root and print its real output."""
    print(f"$ {cmd}\n")
    result = subprocess.run(
        cmd, shell=True, cwd=REPO, capture_output=True, text=True, timeout=timeout
    )
    sys.stdout.write(result.stdout)
    if result.stderr:
        sys.stdout.write(result.stderr)
    return result


def wait_until(cmd, contains, timeout=180, interval=5, tail_chars=1500):
    """Poll `cmd` until its output contains `contains`, printing progress."""
    start = time.time()
    while time.time() - start < timeout:
        r = subprocess.run(cmd, shell=True, cwd=REPO, capture_output=True, text=True)
        if contains in r.stdout:
            elapsed = int(time.time() - start)
            print(f"condition met after ~{elapsed}s\n")
            print(r.stdout[-tail_chars:])
            return elapsed
        time.sleep(interval)
    print(f"timed out after {timeout}s waiting for '{contains}'")
    return None


## 1. Baseline: primary is active, steady-state mirroring is running

In [2]:
run("./scripts/status.sh")


$ ./scripts/status.sh



== Active cluster ==
primary

== Primary cluster broker pods (kafka-primary) ==
NAME                                               READY   STATUS    RESTARTS   AGE
primary-cluster-entity-operator-64df9c9cc6-lqnpz   1/1     Running   0          34m
primary-cluster-primary-pool-0                     1/1     Running   0          36m
primary-cluster-primary-pool-1                     1/1     Running   0          36m
primary-cluster-primary-pool-2                     1/1     Running   0          36m

== DR cluster broker pods (kafka-dr) ==
NAME                                          READY   STATUS    RESTARTS   AGE
dr-cluster-dr-pool-0                          1/1     Running   0          36m
dr-cluster-dr-pool-1                          1/1     Running   0          36m
dr-cluster-dr-pool-2                          1/1     Running   0          36m
dr-cluster-entity-operator-6977d88667-k6kmc   1/1     Running   0          34m

== Producer/consumer pods (kafka-clients) ==
NAME              

CompletedProcess(args='./scripts/status.sh', returncode=0, stdout="== Active cluster ==\nprimary\n\n== Primary cluster broker pods (kafka-primary) ==\nNAME                                               READY   STATUS    RESTARTS   AGE\nprimary-cluster-entity-operator-64df9c9cc6-lqnpz   1/1     Running   0          34m\nprimary-cluster-primary-pool-0                     1/1     Running   0          36m\nprimary-cluster-primary-pool-1                     1/1     Running   0          36m\nprimary-cluster-primary-pool-2                     1/1     Running   0          36m\n\n== DR cluster broker pods (kafka-dr) ==\nNAME                                          READY   STATUS    RESTARTS   AGE\ndr-cluster-dr-pool-0                          1/1     Running   0          36m\ndr-cluster-dr-pool-1                          1/1     Running   0          36m\ndr-cluster-dr-pool-2                          1/1     Running   0          36m\ndr-cluster-entity-operator-6977d88667-k6kmc   1/1     Running

In [3]:
run("microk8s kubectl logs -n kafka-clients deploy/producer --tail=5")
run("microk8s kubectl logs -n kafka-clients deploy/consumer --tail=5")


$ microk8s kubectl logs -n kafka-clients deploy/producer --tail=5



[2026-08-17T18:20:40] produced cluster=primary partition=1 offset=1655 seq=1972
[2026-08-17T18:20:41] produced cluster=primary partition=1 offset=1656 seq=1973
[2026-08-17T18:20:42] produced cluster=primary partition=1 offset=1657 seq=1974
[2026-08-17T18:20:43] produced cluster=primary partition=1 offset=1658 seq=1975
[2026-08-17T18:20:44] produced cluster=primary partition=1 offset=1659 seq=1976
$ microk8s kubectl logs -n kafka-clients deploy/consumer --tail=5



[2026-08-17T18:20:40] consumed cluster=primary partition=1 offset=1656 produced_by=primary seq=1973
[2026-08-17T18:20:41] consumed cluster=primary partition=1 offset=1657 produced_by=primary seq=1974
[2026-08-17T18:20:42] consumed cluster=primary partition=1 offset=1658 produced_by=primary seq=1975
[2026-08-17T18:20:43] consumed cluster=primary partition=1 offset=1659 produced_by=primary seq=1976
[2026-08-17T18:20:44] consumed cluster=primary partition=1 offset=1660 produced_by=primary seq=1977


CompletedProcess(args='microk8s kubectl logs -n kafka-clients deploy/consumer --tail=5', returncode=0, stdout='[2026-08-17T18:20:40] consumed cluster=primary partition=1 offset=1656 produced_by=primary seq=1973\n[2026-08-17T18:20:41] consumed cluster=primary partition=1 offset=1657 produced_by=primary seq=1974\n[2026-08-17T18:20:42] consumed cluster=primary partition=1 offset=1658 produced_by=primary seq=1975\n[2026-08-17T18:20:43] consumed cluster=primary partition=1 offset=1659 produced_by=primary seq=1976\n[2026-08-17T18:20:44] consumed cluster=primary partition=1 offset=1660 produced_by=primary seq=1977\n', stderr='')

## 2. Disaster: primary is declared lost, dr is promoted to active

`failover.sh` stops `mm2-primary-to-dr`, records dr's current per-partition
end offsets as the fail-back watermark, and flips the `ACTIVE_CLUSTER`
ConfigMap to `dr`.


In [4]:
run("./scripts/failover.sh")


$ ./scripts/failover.sh



== Simulating a disaster: primary is declared lost, dr is promoted to active ==
Stopping primary -> dr mirroring (source is being failed over; leaving it running
would let writes made on dr after failover get mirrored back into a dead cluster,
and blocks the reverse leg from being started safely later)...
kafkamirrormaker2.kafka.strimzi.io "mm2-primary-to-dr" deleted from kafka-dr namespace
Recording dr's per-partition end offsets right now — this is the watermark
fail-back will use later to mirror back only what's written to dr AFTER this point,
instead of re-mirroring dr's existing copy of primary's own pre-failover data.
Watermark saved to /home/ozcan/Documents/kafka-mirror/scripts/.failover-watermark.json:
{"offsets":[{"partition":{"cluster":"dr","partition":0,"topic":"drc-demo-topic"},"offset":{"offset":0}},{"partition":{"cluster":"dr","partition":1,"topic":"drc-demo-topic"},"offset":{"offset":1662}},{"partition":{"cluster":"dr","partition":2,"topic":"drc-demo-topic"},"offset":{"o

CompletedProcess(args='./scripts/failover.sh', returncode=0, stdout='== Simulating a disaster: primary is declared lost, dr is promoted to active ==\nStopping primary -> dr mirroring (source is being failed over; leaving it running\nwould let writes made on dr after failover get mirrored back into a dead cluster,\nand blocks the reverse leg from being started safely later)...\nkafkamirrormaker2.kafka.strimzi.io "mm2-primary-to-dr" deleted from kafka-dr namespace\nRecording dr\'s per-partition end offsets right now — this is the watermark\nfail-back will use later to mirror back only what\'s written to dr AFTER this point,\ninstead of re-mirroring dr\'s existing copy of primary\'s own pre-failover data.\nWatermark saved to /home/ozcan/Documents/kafka-mirror/scripts/.failover-watermark.json:\n{"offsets":[{"partition":{"cluster":"dr","partition":0,"topic":"drc-demo-topic"},"offset":{"offset":0}},{"partition":{"cluster":"dr","partition":1,"topic":"drc-demo-topic"},"offset":{"offset":1662}}

In [5]:
# kubelet's ConfigMap sync isn't atomic across pods; poll until the producer
# has actually reconnected to dr.
wait_until(
    "microk8s kubectl logs -n kafka-clients deploy/producer --tail=3",
    "cluster=dr",
    timeout=180,
)


condition met after ~78s

[2026-08-17T18:22:15] produced cluster=dr partition=2 offset=743 seq=2067
[2026-08-17T18:22:16] produced cluster=dr partition=2 offset=744 seq=2068
[2026-08-17T18:22:17] produced cluster=dr partition=2 offset=745 seq=2069



78

In [6]:
wait_until(
    "microk8s kubectl logs -n kafka-clients deploy/consumer --tail=3",
    "cluster=dr",
    timeout=180,
)


condition met after ~0s

[2026-08-17T18:22:15] consumed cluster=dr partition=2 offset=744 produced_by=dr seq=2068
[2026-08-17T18:22:16] consumed cluster=dr partition=2 offset=745 produced_by=dr seq=2069
[2026-08-17T18:22:17] consumed cluster=dr partition=2 offset=746 produced_by=dr seq=2070



0

In [7]:
run("./scripts/status.sh")


$ ./scripts/status.sh



== Active cluster ==
dr

== Primary cluster broker pods (kafka-primary) ==
NAME                                               READY   STATUS    RESTARTS   AGE
primary-cluster-entity-operator-64df9c9cc6-lqnpz   1/1     Running   0          36m
primary-cluster-primary-pool-0                     1/1     Running   0          38m
primary-cluster-primary-pool-1                     1/1     Running   0          38m
primary-cluster-primary-pool-2                     1/1     Running   0          38m

== DR cluster broker pods (kafka-dr) ==
NAME                                          READY   STATUS    RESTARTS   AGE
dr-cluster-dr-pool-0                          1/1     Running   0          38m
dr-cluster-dr-pool-1                          1/1     Running   0          38m
dr-cluster-dr-pool-2                          1/1     Running   0          38m
dr-cluster-entity-operator-6977d88667-k6kmc   1/1     Running   0          36m

== Producer/consumer pods (kafka-clients) ==
NAME                   

CompletedProcess(args='./scripts/status.sh', returncode=0, stdout="== Active cluster ==\ndr\n\n== Primary cluster broker pods (kafka-primary) ==\nNAME                                               READY   STATUS    RESTARTS   AGE\nprimary-cluster-entity-operator-64df9c9cc6-lqnpz   1/1     Running   0          36m\nprimary-cluster-primary-pool-0                     1/1     Running   0          38m\nprimary-cluster-primary-pool-1                     1/1     Running   0          38m\nprimary-cluster-primary-pool-2                     1/1     Running   0          38m\n\n== DR cluster broker pods (kafka-dr) ==\nNAME                                          READY   STATUS    RESTARTS   AGE\ndr-cluster-dr-pool-0                          1/1     Running   0          38m\ndr-cluster-dr-pool-1                          1/1     Running   0          38m\ndr-cluster-dr-pool-2                          1/1     Running   0          38m\ndr-cluster-entity-operator-6977d88667-k6kmc   1/1     Running   0 

## 3. dr serving traffic on its own

At this point `primary-cluster` is considered lost. `dr-cluster` is fully
active, with no mirror running, so everything written now only exists on dr
until a fail-back resync brings it back.


In [8]:
# let some genuinely new dr-origin data accumulate
time.sleep(30)
run("microk8s kubectl logs -n kafka-clients deploy/producer --tail=8")


$ microk8s kubectl logs -n kafka-clients deploy/producer --tail=8



[2026-08-17T18:23:10] produced cluster=dr partition=2 offset=798 seq=2122
[2026-08-17T18:23:11] produced cluster=dr partition=2 offset=799 seq=2123
[2026-08-17T18:23:12] produced cluster=dr partition=2 offset=800 seq=2124
[2026-08-17T18:23:13] produced cluster=dr partition=2 offset=801 seq=2125
[2026-08-17T18:23:14] produced cluster=dr partition=2 offset=802 seq=2126
[2026-08-17T18:23:15] produced cluster=dr partition=2 offset=803 seq=2127
[2026-08-17T18:23:16] produced cluster=dr partition=2 offset=804 seq=2128
[2026-08-17T18:23:17] produced cluster=dr partition=2 offset=805 seq=2129


CompletedProcess(args='microk8s kubectl logs -n kafka-clients deploy/producer --tail=8', returncode=0, stdout='[2026-08-17T18:23:10] produced cluster=dr partition=2 offset=798 seq=2122\n[2026-08-17T18:23:11] produced cluster=dr partition=2 offset=799 seq=2123\n[2026-08-17T18:23:12] produced cluster=dr partition=2 offset=800 seq=2124\n[2026-08-17T18:23:13] produced cluster=dr partition=2 offset=801 seq=2125\n[2026-08-17T18:23:14] produced cluster=dr partition=2 offset=802 seq=2126\n[2026-08-17T18:23:15] produced cluster=dr partition=2 offset=803 seq=2127\n[2026-08-17T18:23:16] produced cluster=dr partition=2 offset=804 seq=2128\n[2026-08-17T18:23:17] produced cluster=dr partition=2 offset=805 seq=2129\n', stderr='')

In [9]:
run(
    "microk8s kubectl exec dr-cluster-dr-pool-0 -n kafka-dr -- "
    "bin/kafka-get-offsets.sh --bootstrap-server localhost:9092 --topic drc-demo-topic"
)


$ microk8s kubectl exec dr-cluster-dr-pool-0 -n kafka-dr -- bin/kafka-get-offsets.sh --bootstrap-server localhost:9092 --topic drc-demo-topic



drc-demo-topic:0:0
drc-demo-topic:1:1662
drc-demo-topic:2:820


CompletedProcess(args='microk8s kubectl exec dr-cluster-dr-pool-0 -n kafka-dr -- bin/kafka-get-offsets.sh --bootstrap-server localhost:9092 --topic drc-demo-topic', returncode=0, stdout='drc-demo-topic:0:0\ndrc-demo-topic:1:1662\ndrc-demo-topic:2:820\n', stderr='')

## 4. Primary recovers: start the fail-back resync

`start-failback-resync.sh` brings up the reverse mirror leg (`mm2-dr-to-primary`)
against a topic filter that matches nothing real first, corrects its starting
offsets to the watermark captured at failover time, then switches it to the
real topic and resumes — see the [runbook appendix](../docs/dr-runbook.md#appendix-the-fail-back-duplication-finding)
for why this needs three phases instead of one.


In [10]:
run("./scripts/start-failback-resync.sh", timeout=240)


$ ./scripts/start-failback-resync.sh



== Starting fail-back resync: dr -> primary ==
Phase 1/3: bringing up the connector against a no-op topic filter (safe regardless of timing)...
kafkamirrormaker2.kafka.strimzi.io/mm2-dr-to-primary created
Waiting for the source connector to settle into STOPPED...
  attempt 1: connector state=STOPPED

Phase 2/3: correcting starting offsets to the failover watermark
(this is safe now: the connector has never matched a real topic yet):
{"offsets":[{"partition":{"cluster":"dr","partition":0,"topic":"drc-demo-topic"},"offset":{"offset":0}},{"partition":{"cluster":"dr","partition":1,"topic":"drc-demo-topic"},"offset":{"offset":1662}},{"partition":{"cluster":"dr","partition":2,"topic":"drc-demo-topic"},"offset":{"offset":743}}]}
{"message":"The offsets for this connector have been altered successfully"}

Phase 3/3: switching to the real topic filter and resuming...
kafkamirrormaker2.kafka.strimzi.io/mm2-dr-to-primary configured
Setting the connector's desired state to running (both live, via 

CompletedProcess(args='./scripts/start-failback-resync.sh', returncode=0, stdout='== Starting fail-back resync: dr -> primary ==\nPhase 1/3: bringing up the connector against a no-op topic filter (safe regardless of timing)...\nkafkamirrormaker2.kafka.strimzi.io/mm2-dr-to-primary created\nWaiting for the source connector to settle into STOPPED...\n  attempt 1: connector state=STOPPED\n\nPhase 2/3: correcting starting offsets to the failover watermark\n(this is safe now: the connector has never matched a real topic yet):\n{"offsets":[{"partition":{"cluster":"dr","partition":0,"topic":"drc-demo-topic"},"offset":{"offset":0}},{"partition":{"cluster":"dr","partition":1,"topic":"drc-demo-topic"},"offset":{"offset":1662}},{"partition":{"cluster":"dr","partition":2,"topic":"drc-demo-topic"},"offset":{"offset":743}}]}\n{"message":"The offsets for this connector have been altered successfully"}\n\nPhase 3/3: switching to the real topic filter and resuming...\nkafkamirrormaker2.kafka.strimzi.io/

In [11]:
# give the resync a little time to make visible progress
time.sleep(20)
run(
    "microk8s kubectl exec dr-cluster-dr-pool-0 -n kafka-dr -- "
    "bin/kafka-get-offsets.sh --bootstrap-server localhost:9092 --topic drc-demo-topic"
)
run(
    "microk8s kubectl exec primary-cluster-primary-pool-0 -n kafka-primary -- "
    "bin/kafka-get-offsets.sh --bootstrap-server localhost:9092 --topic drc-demo-topic"
)


$ microk8s kubectl exec dr-cluster-dr-pool-0 -n kafka-dr -- bin/kafka-get-offsets.sh --bootstrap-server localhost:9092 --topic drc-demo-topic



drc-demo-topic:0:0
drc-demo-topic:1:1662
drc-demo-topic:2:938
$ microk8s kubectl exec primary-cluster-primary-pool-0 -n kafka-primary -- bin/kafka-get-offsets.sh --bootstrap-server localhost:9092 --topic drc-demo-topic



drc-demo-topic:0:0
drc-demo-topic:1:5000
drc-demo-topic:2:1093


CompletedProcess(args='microk8s kubectl exec primary-cluster-primary-pool-0 -n kafka-primary -- bin/kafka-get-offsets.sh --bootstrap-server localhost:9092 --topic drc-demo-topic', returncode=0, stdout='drc-demo-topic:0:0\ndrc-demo-topic:1:5000\ndrc-demo-topic:2:1093\n', stderr='')

## 5. Complete the fail-back: primary becomes active again

In [12]:
run("./scripts/complete-failback.sh")


$ ./scripts/complete-failback.sh



== Completing fail-back: primary becomes active again ==
configmap/active-cluster-config patched
ACTIVE_CLUSTER set to primary.
Removing the dr -> primary resync leg (its job is done)...
kafkamirrormaker2.kafka.strimzi.io "mm2-dr-to-primary" deleted from kafka-primary namespace
Restoring steady-state mirroring: primary -> dr...
kafkamirrormaker2.kafka.strimzi.io/mm2-primary-to-dr created

Fail-back complete. primary is active, dr is on standby, mirroring runs primary -> dr again.
Run ./status.sh to confirm.


CompletedProcess(args='./scripts/complete-failback.sh', returncode=0, stdout='== Completing fail-back: primary becomes active again ==\nconfigmap/active-cluster-config patched\nACTIVE_CLUSTER set to primary.\nRemoving the dr -> primary resync leg (its job is done)...\nkafkamirrormaker2.kafka.strimzi.io "mm2-dr-to-primary" deleted from kafka-primary namespace\nRestoring steady-state mirroring: primary -> dr...\nkafkamirrormaker2.kafka.strimzi.io/mm2-primary-to-dr created\n\nFail-back complete. primary is active, dr is on standby, mirroring runs primary -> dr again.\nRun ./status.sh to confirm.\n', stderr='')

In [13]:
# the consumer rejoins cg-drc on primary and immediately walks into the
# records the resync replayed there — watch it drop every one of them
wait_until(
    "microk8s kubectl logs -n kafka-clients deploy/consumer --tail=5",
    "cluster=primary",
    timeout=180,
)


condition met after ~63s

[2026-08-17T18:26:48] dropped duplicate cluster=primary partition=1 offset=4493 produced_by=primary seq=1472
[2026-08-17T18:26:48] dropped duplicate cluster=primary partition=1 offset=4494 produced_by=primary seq=1473
[2026-08-17T18:26:48] dropped duplicate cluster=primary partition=1 offset=4495 produced_by=primary seq=1474
[2026-08-17T18:26:48] dropped duplicate cluster=primary partition=1 offset=4496 produced_by=primary seq=1475
[2026-08-17T18:26:48] dropped duplicate cluster=primary partition=1 offset=4497 produced_by=primary seq=1476



63

In [14]:
time.sleep(10)
run("microk8s kubectl logs -n kafka-clients deploy/consumer --tail=200 | grep -c 'dropped duplicate'")


$ microk8s kubectl logs -n kafka-clients deploy/consumer --tail=200 | grep -c 'dropped duplicate'



185


CompletedProcess(args="microk8s kubectl logs -n kafka-clients deploy/consumer --tail=200 | grep -c 'dropped duplicate'", returncode=0, stdout='185\n', stderr='')

In [15]:
run("microk8s kubectl logs -n kafka-clients deploy/consumer --tail=15")


$ microk8s kubectl logs -n kafka-clients deploy/consumer --tail=15



[2026-08-17T18:26:48] consumed cluster=primary partition=1 offset=5017 produced_by=primary seq=2337
[2026-08-17T18:26:48] consumed cluster=primary partition=1 offset=5018 produced_by=primary seq=2338
[2026-08-17T18:26:48] consumed cluster=primary partition=1 offset=5019 produced_by=primary seq=2339
[2026-08-17T18:26:48] consumed cluster=primary partition=1 offset=5020 produced_by=primary seq=2340
[2026-08-17T18:26:48] consumed cluster=primary partition=1 offset=5021 produced_by=primary seq=2341
[2026-08-17T18:26:49] consumed cluster=primary partition=1 offset=5022 produced_by=primary seq=2342
[2026-08-17T18:26:50] consumed cluster=primary partition=1 offset=5023 produced_by=primary seq=2343
[2026-08-17T18:26:51] consumed cluster=primary partition=1 offset=5024 produced_by=primary seq=2344
[2026-08-17T18:26:52] consumed cluster=primary partition=1 offset=5025 produced_by=primary seq=2345
[2026-08-17T18:26:53] consumed cluster=primary partition=1 offset=5026 produced_by=primary seq=2346


CompletedProcess(args='microk8s kubectl logs -n kafka-clients deploy/consumer --tail=15', returncode=0, stdout='[2026-08-17T18:26:48] consumed cluster=primary partition=1 offset=5017 produced_by=primary seq=2337\n[2026-08-17T18:26:48] consumed cluster=primary partition=1 offset=5018 produced_by=primary seq=2338\n[2026-08-17T18:26:48] consumed cluster=primary partition=1 offset=5019 produced_by=primary seq=2339\n[2026-08-17T18:26:48] consumed cluster=primary partition=1 offset=5020 produced_by=primary seq=2340\n[2026-08-17T18:26:48] consumed cluster=primary partition=1 offset=5021 produced_by=primary seq=2341\n[2026-08-17T18:26:49] consumed cluster=primary partition=1 offset=5022 produced_by=primary seq=2342\n[2026-08-17T18:26:50] consumed cluster=primary partition=1 offset=5023 produced_by=primary seq=2343\n[2026-08-17T18:26:51] consumed cluster=primary partition=1 offset=5024 produced_by=primary seq=2344\n[2026-08-17T18:26:52] consumed cluster=primary partition=1 offset=5025 produced_

## 6. Final status: back to steady state

In [16]:
run("./scripts/status.sh")


$ ./scripts/status.sh



== Active cluster ==
primary

== Primary cluster broker pods (kafka-primary) ==
NAME                                               READY   STATUS    RESTARTS   AGE
primary-cluster-entity-operator-64df9c9cc6-lqnpz   1/1     Running   0          41m
primary-cluster-primary-pool-0                     1/1     Running   0          43m
primary-cluster-primary-pool-1                     1/1     Running   0          43m
primary-cluster-primary-pool-2                     1/1     Running   0          43m

== DR cluster broker pods (kafka-dr) ==
NAME                                          READY   STATUS    RESTARTS   AGE
dr-cluster-dr-pool-0                          1/1     Running   0          43m
dr-cluster-dr-pool-1                          1/1     Running   0          43m
dr-cluster-dr-pool-2                          1/1     Running   0          43m
dr-cluster-entity-operator-6977d88667-k6kmc   1/1     Running   0          41m

== Producer/consumer pods (kafka-clients) ==
NAME              

CompletedProcess(args='./scripts/status.sh', returncode=0, stdout="== Active cluster ==\nprimary\n\n== Primary cluster broker pods (kafka-primary) ==\nNAME                                               READY   STATUS    RESTARTS   AGE\nprimary-cluster-entity-operator-64df9c9cc6-lqnpz   1/1     Running   0          41m\nprimary-cluster-primary-pool-0                     1/1     Running   0          43m\nprimary-cluster-primary-pool-1                     1/1     Running   0          43m\nprimary-cluster-primary-pool-2                     1/1     Running   0          43m\n\n== DR cluster broker pods (kafka-dr) ==\nNAME                                          READY   STATUS    RESTARTS   AGE\ndr-cluster-dr-pool-0                          1/1     Running   0          43m\ndr-cluster-dr-pool-1                          1/1     Running   0          43m\ndr-cluster-dr-pool-2                          1/1     Running   0          43m\ndr-cluster-entity-operator-6977d88667-k6kmc   1/1     Running

## Summary

- Failover: the producer and consumer reconnected from `primary-cluster` to
  `dr-cluster` with no pod restart, purely by the `ACTIVE_CLUSTER` ConfigMap
  changing and each pod noticing on its own kubelet sync cycle.
- While `dr-cluster` was active, it kept serving new writes completely
  independently of `primary-cluster` — the two clusters share no
  infrastructure.
- The fail-back resync correctly copied dr's new data back to primary, but
  (as documented in the runbook) it also replayed the pre-failover backlog
  once, because of how Kafka Connect's connector lifecycle interacts with
  Strimzi's offset-management flow. The consumer's per-origin dedup caught
  every one of those replayed records.
- The full round trip — primary active, failover to dr, dr active, fail-back
  resync, primary active again — completed with the mirror back in its
  normal steady state and no data loss, only a bounded, correctly-handled
  set of duplicates.

Full write-up: [`docs/dr-runbook.md`](../docs/dr-runbook.md) and
[`docs/architecture.md`](../docs/architecture.md).
